# Stage 2 · 整轨迹 DPO + 核验上下文双视图
干净状态 vs 追加主题相关但不提供支持的 context。L_task(two views) + 0.2 DPO + 0.1 stability；β=0.1，容忍区间 0.05。预算耗尽不算自主 STOP。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    # BF16 + LoRA does not use torchao quantization. Colab's old optional
    # torchao package makes PEFT 0.19.1 fail even on ordinary Linear layers.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


In [ ]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


默认仅联调：MODE=full 读取已准备数据，RUN_FULL_STAGE2=False 控制小规模运行。自动寻找 02 最近成功的 Stage 1 试跑检查点；最多审核 4 道有真实偏好轨迹的训练题，每题最多 2 段（最多 8 次段落审核，底层网络重试另计），最多 2 对完整轨迹和 2 步训练。原始审核缓存可复用，部分上下文包与参考概率写到独立 stage2_probe 目录，不覆盖正式上下文包、参考或模型。审核可用 CPU；训练使用 A100。无合格段落时停止并检查原因，不强行构造稳定性对。设 RUN_FULL_STAGE2=True 会执行全量审核和正式训练，且要求正式 Stage 1 检查点；目前请保持 False。

In [ ]:
import copy, importlib, torch
from datetime import datetime, timezone
from uuid import uuid4
from sa_jev import model, training, experiments, context_views, probes
for module in (model, training, experiments, context_views, probes):
    importlib.reload(module)
RUN_FULL_STAGE2 = False
PROBE_STEPS = 2
PROBE_QUESTIONS = 4
PROBE_PAIRS = 2
STAGE1_CHECKPOINT = None  # None selects the most recent completed 02 probe.
probe_config = copy.deepcopy(config)
probe_config['model'].update(branch_microbatch=1, decision_batch_size=8)
probe_config['training'].update(accumulation=2, state_batch_size=1,
                                validate_every=1, save_every=1)
gpu_gib = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else None
probe_config['training']['memory_limit_gib'] = min(38.0, gpu_gib-2.0) if gpu_gib else 38.0
probe_config.setdefault('context_views', {})['max_candidates'] = 2
if not RUN_FULL_STAGE2:
    STAGE1_CHECKPOINT = Path(STAGE1_CHECKPOINT) if STAGE1_CHECKPOINT else probes.latest_stage1_probe(ROOT)
    probe_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
    PROBE_OUTPUT = ROOT/'runs/stage2_probe'/probe_id/'seed_17'
    print('Source data:', ROOT, flush=True)
    print('Stage 1 probe checkpoint:', STAGE1_CHECKPOINT, flush=True)
    print('Stage 2 probe output:', PROBE_OUTPUT, flush=True)
    print('Limits: questions=', PROBE_QUESTIONS, 'pairs=', PROBE_PAIRS, 'updates=', PROBE_STEPS, flush=True)
else:
    print('FORMAL STAGE 2: full context review, full references, and formal Stage-1 checkpoints required.', flush=True)


In [ ]:
if RUN_FULL_STAGE2:
    from sa_jev.context_views import prepare_context_views
    context_summary = prepare_context_views(ROOT, config)
else:
    context_summary = probes.prepare_stage2_probe(
        ROOT, probe_config, PROBE_OUTPUT, STAGE1_CHECKPOINT,
        question_limit=PROBE_QUESTIONS,
    )
context_summary


In [ ]:
assert torch.cuda.is_available(), 'Stage-2 model training requires a GPU runtime; review preparation can run on CPU.'
if RUN_FULL_STAGE2:
    from sa_jev.experiments import train_variant
    for seed in config['seeds']:
        train_variant(ROOT, config, 'full', seed)
else:
    history = probes.run_stage2_probe(
        ROOT, probe_config, PROBE_OUTPUT, STAGE1_CHECKPOINT,
        max_steps=PROBE_STEPS, pair_limit=PROBE_PAIRS, validation_limit=2,
    )
    print('Stage-2 probe completed; inspect probe_checks.json. These weights are for pipeline checks only.', flush=True)


冻结参考和 DPO 均保留干净轨迹，旧 preferences.jsonl 可复用。两视图使用相同标签，只有普通 context 字段追加已审核文本，不显示“无关”标记或审核信息。无合格段落、未知标签或追加后超出 token 上限的状态只做干净任务训练；不截断证据，不把真实状态变化当作不变对。training.jsonl 记录 context_pairs、unpaired_states、context_over_budget 和各稳定性有效数。旧序列化控制可设 config['training']['context_view_mode']='serialization'，须使用独立实验目录，不能混合续训。